# Calculate effective coverage by quintile and scenario

Also by age, sex, and pregnancy status (though coverage will not vary by pregnancy status due to a lack of data).

This is similar to what the pregnancy simulation does at the individual level, but using groups instead.
It can be shared between multiplication models that do not incorporate individual heterogeneity.

In [1]:
import pandas as pd

In [2]:
location = "nigeria"
vehicle = "rice"
fortificant = "iron"

In [3]:
# Parameters
location = "india"
fortificant = "iron"
vehicle = "rice"


In [4]:
results_dir = f"../results/{fortificant}/{vehicle}"

In [5]:
full_coverage_probability = pd.read_csv(
    f"{results_dir}/baseline_fortification/full_coverage/{location}.csv"
)
full_coverage_probability = full_coverage_probability.set_index(
    [c for c in full_coverage_probability.columns if c != "value"]
).value
full_coverage_probability

sex     age_start  age_end  wealth_quintile  vehicle_name
Female  0          5        1                rice            0.173331
                            2                rice            0.198508
                            3                rice            0.200478
                            4                rice            0.146602
                            5                rice            0.073288
        5          15       1                rice            0.191211
                            2                rice            0.216787
                            3                rice            0.199139
                            4                rice            0.156480
                            5                rice            0.081990
        15         30       1                rice            0.179032
                            2                rice            0.216343
                            3                rice            0.196778
                            4   

In [6]:
any_coverage_probability = pd.read_csv(
    f"{results_dir}/baseline_fortification/any_coverage/{location}.csv"
)
any_coverage_probability = any_coverage_probability.set_index(
    [c for c in any_coverage_probability.columns if c != "value"]
).value
any_coverage_probability

sex     age_start  age_end  wealth_quintile  vehicle_name
Female  0          5        1                rice            0.610227
                            2                rice            0.603021
                            3                rice            0.562766
                            4                rice            0.519586
                            5                rice            0.333848
        5          15       1                rice            0.722657
                            2                rice            0.691105
                            3                rice            0.639364
                            4                rice            0.577194
                            5                rice            0.364202
        15         30       1                rice            0.649086
                            2                rice            0.636175
                            3                rice            0.574530
                            4   

In [7]:
partial_coverage_mean = pd.read_csv(
    f"{results_dir}/baseline_fortification/partial_coverage_amount/mean/{location}.csv"
)
partial_coverage_mean = partial_coverage_mean.set_index(
    [c for c in partial_coverage_mean.columns if c != "value"]
).value
partial_coverage_mean

sex     age_start  age_end  wealth_quintile  vehicle_name
Female  0          5        1                rice            0.524382
                            2                rice            0.552721
                            3                rice            0.551125
                            4                rice            0.530210
                            5                rice            0.471885
        5          15       1                rice            0.566689
                            2                rice            0.565275
                            3                rice            0.560457
                            4                rice            0.535734
                            5                rice            0.477592
        15         30       1                rice            0.535427
                            2                rice            0.563819
                            3                rice            0.558023
                            4   

In [8]:
current_coverage = (
    full_coverage_probability
    + (any_coverage_probability - full_coverage_probability) * partial_coverage_mean
)
current_coverage

sex     age_start  age_end  wealth_quintile  vehicle_name
Female  0          5        1                rice            0.402431
                            2                rice            0.422091
                            3                rice            0.400144
                            4                rice            0.344362
                            5                rice            0.196243
        5          15       1                rice            0.492376
                            2                rice            0.484907
                            3                rice            0.445866
                            4                rice            0.381871
                            5                rice            0.216772
        15         30       1                rice            0.430712
                            2                rice            0.453052
                            3                rice            0.407573
                            4   

In [9]:
scenarios = {
    "india": ["intervention"],
    "nigeria": ["intervention"],
    "ethiopia": ["intervention_25_nrv", "intervention_100_nrv", "intervention_45_ppm"],
}[location]

In [10]:
any_consumption = pd.read_csv(
    f"../results/{vehicle}/vehicle_consumption/any/{location}.csv"
)
any_consumption = any_consumption.set_index(
    [c for c in any_consumption.columns if c != "value"]
).value
any_consumption

sex     age_start  age_end  wealth_quintile  vehicle_name
Female  0          5        1                rice            0.919076
                            2                rice            0.887756
                            3                rice            0.875160
                            4                rice            0.874946
                            5                rice            0.879201
        5          15       1                rice            0.987435
                            2                rice            0.971206
                            3                rice            0.976108
                            4                rice            0.985006
                            5                rice            0.988478
        15         30       1                rice            0.989477
                            2                rice            0.962284
                            3                rice            0.949051
                            4   

In [11]:
if (
    len(
        any_consumption.reset_index()[["sex", "age_start", "age_end"]].drop_duplicates()
    )
    == 1
):
    # Assumed same for all ages/sexes
    any_consumption = any_consumption.droplevel(["sex", "age_start", "age_end"])
    display(any_consumption)

In [12]:
fortifiability = pd.read_csv(
    f"../results/{vehicle}/vehicle_consumption/fortifiability/{location}.csv"
)
fortifiability = fortifiability.set_index(
    [c for c in fortifiability.columns if c != "value"]
).value
fortifiability

sex     age_start  age_end  wealth_quintile  vehicle_name
Female  0          5        1                rice            0.740639
                            2                rice            0.764498
                            3                rice            0.737789
                            4                rice            0.663464
                            5                rice            0.421257
        5          15       1                rice            0.840406
                            2                rice            0.833037
                            3                rice            0.791817
                            4                rice            0.714459
                            5                rice            0.458717
        15         30       1                rice            0.774598
                            2                rice            0.799744
                            3                rice            0.746989
                            4   

In [13]:
import pathlib, numpy as np

for scenario in scenarios:
    intervention_coverage = pd.read_csv(
        f"{results_dir}/{scenario}/intervention_fortification/any_coverage/{location}.csv"
    )
    intervention_coverage = intervention_coverage.set_index(
        [c for c in intervention_coverage.columns if c != "value"]
    ).value
    target_coverage = intervention_coverage * fortifiability
    display(target_coverage)
    assert (
        (target_coverage > current_coverage.reindex_like(target_coverage))
        | np.isclose(target_coverage, current_coverage.reindex_like(target_coverage))
    ).all()
    target_coverage[
        np.isclose(target_coverage, current_coverage.reindex_like(target_coverage))
    ] = current_coverage.reindex_like(target_coverage)
    # Not all coverage is effective -- this is as a proportion of coverage!
    effectiveness = pd.read_csv(
        f"{results_dir}/{scenario}/intervention_fortification/effectiveness/{location}.csv"
    )
    effectiveness = effectiveness.set_index(
        [c for c in effectiveness.columns if c != "value"]
    ).value
    effective_intervention_coverage = any_consumption * target_coverage * effectiveness
    path = f"{results_dir}/{scenario}/intervention_fortification/effective_coverage/{location}.csv"
    pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
    effective_intervention_coverage.reset_index().to_csv(path, index=False)

sex     age_start  age_end  wealth_quintile  vehicle_name
Female  0          5        1                rice            0.681388
                            2                rice            0.703338
                            3                rice            0.678766
                            4                rice            0.610387
                            5                rice            0.387556
        5          15       1                rice            0.773174
                            2                rice            0.766394
                            3                rice            0.728472
                            4                rice            0.657303
                            5                rice            0.422019
        15         30       1                rice            0.712630
                            2                rice            0.735765
                            3                rice            0.687230
                            4   

In [14]:
# Not all coverage is effective -- this is as a proportion of coverage!
effectiveness = pd.read_csv(
    f"{results_dir}/baseline_fortification/effectiveness/{location}.csv"
)
effectiveness = effectiveness.set_index(
    [c for c in effectiveness.columns if c != "value"]
).value

In [15]:
effective_baseline_coverage = any_consumption * current_coverage * effectiveness
effective_baseline_coverage

wealth_quintile  vehicle_name  sex     age_start  age_end
1                rice          Female  0          5          0.295892
                                       5          15         0.388951
                                       15         30         0.340944
                                       30         50         0.368677
                                       50         125        0.385878
                               Male    0          5          0.302019
                                       5          15         0.392069
                                       15         30         0.363008
                                       30         50         0.355872
                                       50         125        0.380607
2                rice          Female  0          5          0.299771
                                       5          15         0.376756
                                       15         30         0.348772
                                

In [16]:
path = f"{results_dir}/baseline_fortification/effective_coverage/{location}.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
effective_baseline_coverage.reset_index().to_csv(path, index=False)